# MIAAM — per-student sequence preparation

This notebook processes the MIAAM/NeurIPS dataset of primary-school students completing adaptive math exercises and produces per-student interaction sequences for Knowledge Tracing experiments.

It outputs three parquet files into `../data/`:
- `sequences.parquet` — full dataset (~38 k students), one row per student; attempts stored as a chronologically ordered list of structs
- `train_sequences.parquet` — 80 % random split (seed 42)
- `val_sequences.parquet` — 20 % random split (seed 42)

Each attempt struct carries: `exercise_id_int`, `exercise_id`, `activity_id_int`, `activity_id`, `data_correct`, `created_at` (Unix seconds), `data_duration`, `work_mode`, `data_answer`, `session_id`, `module_name`.

## 1) Load MIAAM dataset

Loads the three source files from `../HF_neurips_official_dataset/`:
- `maths_data_filtered.parquet` — 36,837 students, 6,656,038 attempts (students with <5 attempts or adaptive-test-only histories excluded)
- `maths_exercises_table.parquet` — exercise metadata including hierarchy IDs (`objective_id`, `activity_id`, `module_id`) and text content
- `maths_dependencies.json` — activity-level pedagogical dependency graph

In [ ]:

import polars as pl
import json
import pathlib
import os
import numpy as np
from exercise_embedding import encode_all
from attempt_embedding import encode_interactions
from tqdm import tqdm


DATASET = pathlib.Path("../../MIAAM")
SAVE_FOLDER = pathlib.Path("../data")

interactions = pl.read_parquet(DATASET / "data/maths_data_filtered.parquet")
exercises    = pl.read_parquet(DATASET / "data/maths_exercises_table.parquet")

print(interactions.shape, exercises.shape)

(6481693, 14) (7151, 12)


In [69]:
interactions.head(2)

user_id,classroom_id,playlist_or_module_id,exercise_id,created_at,data_correct,work_mode,data_answer,data_duration,source,attempt_index,session_id,module_id,module_name
str,str,str,str,datetime[μs],bool,str,str,f64,str,u32,str,str,str
"""36705bac-0394-4d0a-8365-3be80a…","""58f3c4d4-01dc-4be9-8435-f7f46f…","""63e98e5f-94e3-4630-9704-076882…","""97516718-fd8f-4f90-8ead-56dff6…",2023-12-01 00:00:12.815,true,"""zpdes""","""[1]""",2335.0,"""am""",1,"""am::36705bac-0394-4d0a-8365-3b…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul"""
"""36705bac-0394-4d0a-8365-3be80a…","""58f3c4d4-01dc-4be9-8435-f7f46f…","""63e98e5f-94e3-4630-9704-076882…","""f90ab87b-7054-4de8-b836-300653…",2023-11-24 00:01:50.092,true,"""zpdes""","""[1]""",3508.0,"""am""",1,"""am::36705bac-0394-4d0a-8365-3b…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul"""


In [70]:
exercises.head(2)

exercise_id,gameplay_type,content,module_id,module_name,objective_id,objective_name,objective_pedagogical_intent,activity_id,activity_name,activity_pedagogical_intent,source
str,str,str,str,str,str,str,str,str,str,str,str
"""39676249-728a-4d21-9a58-c31451…","""INTERVAL_COLORING""","""{""instruction"": ""[description …","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul""","""5d6e6d4b-b905-41c1-81d6-024fbe…","""Découvrir la multiplication""","""En utilisant la ligne numériqu…","""8e532d14-06dc-45db-a1c9-bdf4c7…","""Toutes tables""","""Décomposer toutes les tables s…","""am"""
"""1c921f06-1005-11ed-861d-0242ac…","""MULTIPLE_CHOICE""","""{""instruction"": ""Clique sur le…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul""","""57c36436-8e30-418d-9656-b888de…","""Comparer des nombres (1-100)""","""Comparer des quantités (chiffr…","""a8e9e045-c72f-4722-bb0a-4173d0…","""Nombres entre 60 et 99 (chiffr…","""Comparer des nombres (représen…","""am"""


### Filter exercises without a screenshot

Some exercises in `maths_exercises_table.parquet` have no corresponding screenshot in `data/screenshots/compressed/`. They are removed from both `exercises` and `interactions` before any further processing, so that all remaining exercises have visual content available for multimodal models.

In [71]:
SCREENSHOTS = DATASET / "data/screenshots/compressed"
screenshot_ids = {
    f.stem
    for source_dir in SCREENSHOTS.iterdir()
    for f in source_dir.iterdir()
    if f.suffix == ".png"
}

missing_screenshots = exercises.filter(~pl.col("exercise_id").is_in(list(screenshot_ids)))
print(f"Exercises without a raw screenshot: {len(missing_screenshots)}")
print(missing_screenshots.select(["exercise_id", "source", "module_name", "activity_name"]))

exercises    = exercises.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))
interactions = interactions.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))
print(f"\nAfter filtering: {interactions.shape[0]} attempts, {interactions['exercise_id'].n_unique()} unique exercises")

Exercises without a raw screenshot: 12
shape: (12, 4)
┌─────────────────────────────────┬────────┬───────────────────┬─────────────────────────────────┐
│ exercise_id                     ┆ source ┆ module_name       ┆ activity_name                   │
│ ---                             ┆ ---    ┆ ---               ┆ ---                             │
│ str                             ┆ str    ┆ str               ┆ str                             │
╞═════════════════════════════════╪════════╪═══════════════════╪═════════════════════════════════╡
│ a84300cf-53ce-497d-aa20-1778f1… ┆ am     ┆ Nombres et calcul ┆ Table de 5                      │
│ fa2d0f28-0e2c-4d85-a1ef-0df785… ┆ am     ┆ Nombres et calcul ┆ Points disposés aléatoirement … │
│ 87fd4c76-c5dc-4cc4-b056-7096c5… ┆ am     ┆ Nombres et calcul ┆ Points disposés aléatoirement … │
│ e5fae336-e143-4070-b7f3-4ce23a… ┆ am     ┆ Nombres et calcul ┆ Table de 7                      │
│ 415808bf-9327-4e3a-a6af-694cf6… ┆ am     ┆ Nombres et

In [72]:
user_id_map = (
    interactions.select("user_id")
    .unique()
    .sort("user_id")
    .with_row_index("user_id_int")
)
interactions = interactions.join(user_id_map, on="user_id", how="left")

In [73]:
exercises_embedding = encode_all(exercises, SCREENSHOTS)

Encoding exercises: 100%|██████████| 893/893 [05:25<00:00,  2.74batch/s]


In [ ]:

interactions_embeddings = encode_interactions(interactions.head(100))
interactions = interactions.head(100)

Encoding interactions: 100%|██████████| 13/13 [00:00<00:00, 28.04batch/s]


In [75]:
# Assign stable row indices matching the interaction_id used by encode_interactions.
# Left joins above preserve row order, so this index is consistent with interactions_embeddings.
interactions_indexed = interactions.with_row_index("interaction_id")

# Join exercise and interaction embeddings onto the attempt log
interactions_full = (
    interactions_indexed
    .join(
        exercises_embedding.rename({"embedding": "exercise_embedding"}),
        on="exercise_id",
        how="left",
    )
    .join(
        interactions_embeddings.rename({"embedding": "interaction_embedding"}),
        on="interaction_id",
        how="left",
    )
)

In [76]:
# Sort chronologically per student, collect into per-student lists
grouped = (
    interactions_full
    .select(["user_id_int", "data_correct", "created_at", "exercise_embedding", "interaction_embedding"])
    .with_columns(pl.col("created_at").dt.epoch(time_unit="s").alias("created_at"))
    .sort(["user_id_int", "created_at"])
    .group_by("user_id_int", maintain_order=True)
    .agg([
        pl.col("exercise_embedding").alias("exercise_embeddings"),
        pl.col("interaction_embedding").alias("interaction_embeddings"),
        pl.col("data_correct").cast(pl.Int8).alias("labels"),
    ])
)

## 2) Build per-student sequences

Enriches the attempt log with integer-encoded IDs and restructures it into per-student sequences suitable as input for Knowledge Tracing models. The result is a Polars DataFrame in parquet format — one row per student, all attempt fields collected into a parallel-list struct column ordered chronologically by `created_at`.

### Build per-student sequences

Group the enriched attempt log by student and collect each field into a parallel list ordered chronologically by `created_at` (converted to a Unix timestamp in seconds). The result is a dict keyed by `user_id_int`, where each value is a dict of lists — one list per field, all of the same length.

In [77]:
# Interleave: history = [ex_emb_0, int_emb_0, ex_emb_1, int_emb_1, ...], shape (2*T, H)
records = []
for row in tqdm(grouped.iter_rows(named=True), total=len(grouped), desc="Building sequences"):
    history = []
    for ex, inter in zip(row["exercise_embeddings"], row["interaction_embeddings"]):
        history.append(list(ex))
        history.append(list(inter))
    records.append({
        "user_id_int": row["user_id_int"],
        "history": history,
        "labels": row["labels"],
    })

dim = len(records[0]["history"][0])
seq_df = pl.DataFrame(
    records,
    schema={
        "user_id_int": pl.UInt32,
        "history": pl.List(pl.Array(pl.Float32, dim)),
        "labels": pl.List(pl.Int8),
    },
)

Building sequences: 100%|██████████| 8/8 [00:00<00:00, 2856.91it/s]


In [78]:
shuffled = seq_df.sample(fraction=1.0, shuffle=True, seed=42)
n_val = int(len(shuffled) * 0.2)

val_df = shuffled[:n_val]
train_df = shuffled[n_val:]

train_df.write_parquet(SAVE_FOLDER / "train_sequences_embedded.parquet")
val_df.write_parquet(SAVE_FOLDER / "val_sequences_embedded.parquet")